<a href="https://colab.research.google.com/github/sethomode44/201/blob/main/Ex11_Optimal_Active_Portfolio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Investments: Theory and Data Analysis**, Bates, Boyer, and Fletcher


# Example Chapter 11: Optimal Active Portfolio
In this example we estimate the optimal active tilt towards the zero-cost portfolio that goes long decile 1 short-term reversal portfolio and short decile 10 short-term reversal portfolio from the Ken French Data Library.


### Imports and Setup

In [1]:
# Import packages
%pip install -q simple-finance
import simple_finance as sf
import requests
import pandas as pd
import numpy as np

# Establish display options when printing DataFrames
pd.set_option('display.max_columns', None)   # Show all columns without truncation
pd.set_option('display.width', 1000)   # Set the display width so output stays on one line
pd.set_option("display.max_rows", 20) # Force truncation if DataFrame has more than 20 rows

### Load in Data Using the Python Function
See Ex07-Ken_French.ipynb for a complete description of the function in the course GiHub repo.


In [2]:
# Get returns on short-term reveral deciles
df=sf.get_ff_strategies('momentum', details=False)
print(df)

          Dec 1   Dec 2   Dec 3   Dec 4   Dec 5   Dec 6   Dec 7   Dec 8   Dec 9  Dec 10  mkt-rf     smb     hml      rf
date                                                                                                                   
1927-01 -0.0332 -0.0454  0.0267 -0.0029 -0.0041  0.0093  0.0030  0.0071 -0.0014 -0.0024 -0.0005 -0.0032  0.0458  0.0025
1927-02  0.0739  0.0601  0.0703  0.0746  0.0434  0.0398  0.0299  0.0320  0.0414  0.0704  0.0417  0.0007  0.0272  0.0026
1927-03 -0.0323 -0.0305 -0.0384 -0.0480 -0.0046 -0.0235  0.0196  0.0049  0.0035  0.0613  0.0014 -0.0177 -0.0238  0.0030
1927-04  0.0128 -0.0301 -0.0244  0.0043  0.0119 -0.0016  0.0203 -0.0074  0.0181  0.0549  0.0047  0.0039  0.0065  0.0025
1927-05  0.0271  0.0454  0.0597  0.0314  0.0614  0.0593  0.0522  0.0632  0.0819  0.0592  0.0545  0.0155  0.0480  0.0030
...         ...     ...     ...     ...     ...     ...     ...     ...     ...     ...     ...     ...     ...     ...
2025-09  0.0313 -0.0233 -0.0010  0.0497 

### Define Inputs
In the code cell below we define the covariance matrix and the vector of expected returns.  Here you can enter your own covriance matrix and vector of expected returns as desired, with any number of securities. You can also load these inputs from your account on Google Drive, or your own GitHub repo.   

In [3]:
df = df.copy()  # avoid SettingWithCopyWarning

# Define the zero-cost portfolio
df['Dec1_Dec10'] = df['Dec 10'] - df['Dec 1']

# Estimate Expected excess returns
expected_returns = df[['mkt-rf', 'Dec1_Dec10']].mean().to_numpy()

# Estimate the covariance matrix
covariance_matrix = df[['mkt-rf', 'Dec1_Dec10']].cov().to_numpy()
print(expected_returns)
print(covariance_matrix)
# Note that the covariance matrix here is defined using the excess market return.
# Subtracting off the risk-free rate, which is close to constant, has very little
# effect on the covariance matrix.  In some aplications is is common to define
# the covariance matrix using the excess market return.

[0.00688764 0.01125643]
[[ 0.00282616 -0.00153263]
 [-0.00153263  0.00625087]]


### Define the Tangent Portfolio
Function: `tangent_portfolio`   

**Inputs**
* `expected_returns`: the array of expected returns
* `covariance_matrix`: the covariance matrix
* `factors`: if `True` then we are finding the optimal benchmark tilt where the benchmark is the first asset in the array of expected returns and asset `1` in the covariance matrix.

**Output**
* `tangent_weights`: the weights of the tangent portfolio
* `tangent_return`: the expected return of the tangent portfolio
* `tangent_volatility`: the volatility of the tangent portfolio


In [5]:
tangent_weights, tangent_return, tangent_volatility = sf.tangent_portfolio(expected_returns, covariance_matrix, factors=True)

print(tangent_weights)
print(tangent_return)
print(tangent_volatility)

[1.         0.70247366]
0.014794985006114744
0.061298499186571154


### Test Trading Strategy

In [6]:
# For comparison with Ex07-Ken_French.ipynb
#tangent_weights = np.array([1, 0.250])

# Create realized excess portfolio returns
excess = df[['mkt-rf', 'Dec1_Dec10']].to_numpy()   # shape (T,2)

# Calculate the excess returns as a 1D array
# @ performs an inner product
excess_array = excess @ tangent_weights

# Convert the Series to a DataFrame
excess = pd.DataFrame(excess_array, index=df.index, columns=['excess'])

# compute total return
total_return = excess['excess'] + df['rf']

# Get monthly mean returns, volatility, and sharpe
# Call .mean() to get a scalar mean from the total_return Series
mean_monthly = total_return.mean()
# Select the 'excess' column before calling .mean() as 'excess' is a DataFrame
mean_excess_monthly  = excess['excess'].mean()
# Select the 'excess' column before calling .std() as 'excess' is a DataFrame
std_monthly  = total_return.std()
sharpe_monthly = mean_excess_monthly / std_monthly

# annualize
ann_return = 12 * mean_monthly
ann_vol    = np.sqrt(12) * std_monthly
ann_sharpe = sharpe_monthly * np.sqrt(12)

# Print results
print(f"Average Return: {ann_return:.3f}")
print(f"Annualized Volatility: {ann_vol:.3f}")
print(f"Annualized Sharpe Ratio: {ann_sharpe:.3f}")

Average Return: 0.210
Annualized Volatility: 0.213
Annualized Sharpe Ratio: 0.835
